# 02 EDA Visualization

Load the final dataset, inspect coverage and missingness, and export the main charts.

## Imports And Load

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

ROOT = Path.cwd()
FINAL_DIR = ROOT / 'data' / 'final'
FIGURES_DIR = ROOT / 'outputs' / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
sns.set_theme(style='whitegrid')
df = pd.read_parquet(FINAL_DIR / 'ml_ready_player_games.parquet')
df.head()

,season,player_id,player_name,team_id,team_abbr,game_id,game_date,home_away,opponent_team,min,...,missing_pts_l5,missing_reb_l5,missing_ast_l5,missing_min_l5,missing_pts_l10,missing_reb_l10,missing_ast_l10,missing_min_l10,missing_usg_l5,missing_usg_l10
0,2023-24,2544,LeBron James,1610612747,LAL,0022300061,2023-10-24,away,DEN,29.010000,...,0.0,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.000,0.000
1,2023-24,2544,LeBron James,1610612747,LAL,0022300076,2023-10-26,home,PHX,35.000000,...,0.0,0.0,0.000000,2.500000,0.0,0.0,0.000000,2.500000,0.259,0.259
2,2023-24,2544,LeBron James,1610612747,LAL,0022300100,2023-10-29,away,SAC,39.083333,...,0.0,0.0,0.000000,2.500000,0.0,0.0,0.000000,2.500000,0.259,0.259
3,2023-24,2544,LeBron James,1610612747,LAL,0022300111,2023-10-30,home,ORL,32.783333,...,8.0,3.0,0.333333,17.296667,8.0,3.0,0.333333,17.296667,0.441,0.441
4,2023-24,2544,LeBron James,1610612747,LAL,0022300127,2023-11-01,home,LAC,42.483333,...,24.5,5.5,4.333333,73.386667,24.5,5.5,4.333333,73.386667,0.548,0.548


## Dataset Overview

In [2]:
overview = {
    'rows': len(df),
    'players': df['player_id'].nunique(),
    'games': df['game_id'].nunique(),
    'seasons': sorted(df['season'].unique().tolist()),
}
pd.Series(overview)

rows                    52707
players                   694
games                    2460
seasons    [2023-24, 2024-25]
dtype: object

## Missingness

In [3]:
missing = df.isna().mean().sort_values(ascending=False).rename('missing_pct').reset_index().rename(columns={'index': 'column'})
missing.head(25)

,column,missing_pct
0,l10_avg_reb,0.021648
1,l5_avg_reb,0.021648
2,season_avg_min,0.021648
3,l10_avg_ast,0.021648
4,season_avg_pts,0.021648
5,l5_avg_pts,0.021648
6,l5_avg_fg3a,0.021648
7,l10_avg_fg3a,0.021648
8,l5_avg_fta,0.021648
9,season_avg_fta,0.021648


## Descriptive Stats

In [4]:
df[['target_pts', 'target_reb', 'target_ast', 'season_avg_pts', 'l5_avg_pts', 'l10_avg_pts', 'opponent_pace', 'opponent_def_rating']].describe()

,target_pts,target_reb,target_ast,season_avg_pts,l5_avg_pts,l10_avg_pts,opponent_pace,opponent_def_rating
count,52707.000000,52707.000000,52707.000000,51566.000000,51566.000000,51566.000000,52707.000000,52707.000000
mean,10.643178,4.091051,2.483996,10.266545,10.579178,10.510735,99.372206,114.106902
std,8.891614,3.468508,2.640523,6.925551,7.388857,7.165350,1.846628,2.964360
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,95.960000,106.600000
25%,4.000000,2.000000,1.000000,5.000000,4.800000,5.000000,97.930000,112.100000
50%,9.000000,3.000000,2.000000,8.750000,9.000000,9.000000,99.460000,114.400000
75%,16.000000,6.000000,4.000000,13.934694,14.800000,14.600000,100.620000,115.700000
max,73.000000,31.000000,23.000000,41.000000,44.200000,41.000000,103.690000,119.600000


## Target Distributions

In [5]:
for col in ['target_pts', 'target_reb', 'target_ast']:
    fig, ax = plt.subplots(figsize=(8, 5))
    sns.histplot(df[col].dropna(), bins=30, kde=True, ax=ax)
    ax.set_title(f'Distribution of {col}')
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / f'{col}_distribution.png', dpi=180)
    plt.close(fig)
sorted(p.name for p in FIGURES_DIR.glob('*distribution*.png'))

['target_ast_distribution.png',
 'target_pts_distribution.png',
 'target_reb_distribution.png']

## Correlation Heatmap

In [6]:
heat_cols = ['target_pts', 'target_reb', 'target_ast', 'season_avg_pts', 'season_avg_reb', 'season_avg_ast', 'l5_avg_pts', 'l5_avg_reb', 'l5_avg_ast', 'l10_avg_pts', 'l10_avg_reb', 'l10_avg_ast', 'season_avg_min', 'opponent_pace', 'opponent_def_rating', 'teammates_out_count']
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(df[heat_cols].corr(), cmap='coolwarm', center=0, ax=ax)
ax.set_title('Correlation Heatmap')
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'correlation_heatmap.png', dpi=180)
plt.close(fig)
FIGURES_DIR / 'correlation_heatmap.png'

WindowsPath('c:/Users/aruls/anaconda_projects/analytics/NBACapstone-main/ipynb/outputs/figures/correlation_heatmap.png')

## Scatter Plots

In [7]:
plot_df = df.sample(min(len(df), 5000), random_state=42)
scatter_specs = [
    ('season_avg_min', 'target_pts'),
    ('season_avg_min', 'target_reb'),
    ('season_avg_min', 'target_ast'),
    ('season_avg_pts', 'target_pts'),
    ('season_avg_reb', 'target_reb'),
    ('season_avg_ast', 'target_ast'),
    ('l5_avg_pts', 'target_pts'),
    ('l5_avg_reb', 'target_reb'),
    ('l5_avg_ast', 'target_ast'),
    ('l10_avg_pts', 'target_pts'),
    ('l10_avg_reb', 'target_reb'),
    ('l10_avg_ast', 'target_ast'),
    ('teammates_out_count', 'target_pts'),
    ('missing_pts_l5', 'target_pts'),
    ('missing_reb_l5', 'target_reb'),
    ('missing_ast_l5', 'target_ast'),
    ('opponent_pace', 'target_pts'),
    ('opponent_def_rating', 'target_pts'),
]
for x, y in scatter_specs:
    fig, ax = plt.subplots(figsize=(7, 5))
    sns.scatterplot(data=plot_df, x=x, y=y, alpha=0.35, s=18, ax=ax)
    ax.set_title(f'{x} vs {y}')
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / f'{x}_vs_{y}.png', dpi=180)
    plt.close(fig)
len(list(FIGURES_DIR.glob('*.png')))

22